# Prova gli hook senza Claude Code
Modulo *Hooks*, esercizio. Gli hook sono script che Claude Code lancia passando un JSON su stdin. Qui il JSON lo scrivi tu: così vedi, senza installare nulla e senza account, cosa risponde ogni hook.

Esegui le celle dall'alto verso il basso (Maiusc+Invio su ognuna).

In [ ]:
%%bash
# jq e lo strumento che gli hook usano per leggere il JSON dell'evento
which jq >/dev/null || (apt-get -qq update && apt-get -qq install -y jq >/dev/null)
which npm >/dev/null || (apt-get -qq update && apt-get -qq install -y nodejs npm >/dev/null)
rm -rf /tmp/hooks-demo && mkdir -p /tmp/hooks-demo/.claude/hooks && cd /tmp/hooks-demo
git init -q . && git config user.email demo@example.com && git config user.name demo
jq --version; npm --version

## 1. Hook che blocca `rm -rf`

In [ ]:
%%writefile /tmp/hooks-demo/.claude/hooks/block-rm.sh
#!/usr/bin/env bash
COMMAND=$(jq -r '.tool_input.command')

if echo "$COMMAND" | grep -q 'rm -rf'; then
  jq -n '{
    hookSpecificOutput: {
      hookEventName: "PreToolUse",
      permissionDecision: "deny",
      permissionDecisionReason: "Comando distruttivo bloccato da un hook di progetto"
    }
  }'
else
  exit 0  # nessuna decisione: si applica il normale flusso dei permessi
fi

In [ ]:
%%bash
cd /tmp/hooks-demo && chmod +x .claude/hooks/*.sh
echo '--- rm -rf build/ ---'
echo '{"tool_input":{"command":"rm -rf build/"}}' | .claude/hooks/block-rm.sh; echo "exit code: $?"
echo '--- ls ---'
echo '{"tool_input":{"command":"ls"}}' | .claude/hooks/block-rm.sh; echo "exit code: $?"

Nel primo caso vedi un JSON con `permissionDecision: deny` e il motivo; nel secondo non stampa nulla (nessuna decisione, si applica il normale flusso dei permessi). In entrambi i casi l'exit code è 0.

## 2. Hook che protegge le migrazioni

In [ ]:
%%writefile /tmp/hooks-demo/.claude/hooks/protect-migrations.sh
#!/usr/bin/env bash
FILE=$(jq -r '.tool_input.file_path // empty')

case "$FILE" in
  */migrations/*|*/migrate/*)
    jq -n --arg f "$FILE" '{
      hookSpecificOutput: {
        hookEventName: "PreToolUse",
        permissionDecision: "deny",
        permissionDecisionReason: ("Le migrazioni non si modificano dopo essere state applicate. " +
          "Per cambiare lo schema, crea una nuova migrazione. File rifiutato: " + $f)
      }
    }'
    ;;
  *)
    exit 0
    ;;
esac

In [ ]:
%%bash
cd /tmp/hooks-demo && chmod +x .claude/hooks/*.sh
echo '--- file in db/migrations ---'
echo '{"tool_input":{"file_path":"/p/db/migrations/001.sql"}}' | .claude/hooks/protect-migrations.sh
echo '--- file in src ---'
echo '{"tool_input":{"file_path":"/p/src/app.js"}}' | .claude/hooks/protect-migrations.sh; echo "exit code: $?"

## 3. Hook che lancia i test e riporta i fallimenti

In [ ]:
%%writefile /tmp/hooks-demo/.claude/hooks/test.sh
#!/usr/bin/env bash
# Nota: qui manca la "-e" (fermati al primo errore) che c'era in lint.sh,
# ed e VOLUTO: i test che falliscono fanno "fallire" npm, ma vogliamo
# gestire noi quel fallimento (leggerne l'output e rimandarlo a Claude),
# non far morire lo script a meta.
set -uo pipefail

FILE=$(jq -r '.tool_input.file_path // empty')
[ -z "$FILE" ] && exit 0

if ! OUTPUT=$(npm test --silent 2>&1); then
  # Solo la coda: l'output completo di un test runner inonderebbe il contesto.
  echo "$OUTPUT" | tail -40 >&2
  exit 2
fi

In [ ]:
%%bash
cd /tmp/hooks-demo && chmod +x .claude/hooks/*.sh
# un "progetto" finto il cui test fallisce
echo '{"name":"demo","scripts":{"test":"echo TEST FALLITO: atteso 200, ricevuto 500; exit 1"}}' > package.json
echo '{"tool_input":{"file_path":"src/app.js"}}' | .claude/hooks/test.sh 2>&1; echo "exit code: $?"

Exit code **2** e l'output del test su stderr: è il segnale con cui Claude Code rimanda l'errore a Claude. Se `package.json` avesse `"test": "exit 0"` l'hook uscirebbe con 0 senza stampare nulla.

## 4. Hook all'avvio della sessione

In [ ]:
%%writefile /tmp/hooks-demo/.claude/hooks/context.sh
#!/usr/bin/env bash
# il nome del branch corrente (o "non-git" fuori da un repository)
BRANCH=$(git branch --show-current 2>/dev/null || echo "non-git")
# quanti file risultano modificati: git status in formato compatto,
# contato con wc -l (tr toglie gli spazi dal numero)
SPORCO=$(git status --porcelain 2>/dev/null | wc -l | tr -d ' ')
# esito dell'ultimo run di CI su GitHub, letto con gh
CI=$(gh run list --limit 1 --json conclusion --jq '.[0].conclusion' 2>/dev/null || echo "sconosciuto")

# costruisce il JSON di risposta: additionalContext è il testo che
# Claude Code inietta nel contesto della sessione appena avviata
jq -n --arg b "$BRANCH" --arg s "$SPORCO" --arg c "$CI" '{
  hookSpecificOutput: {
    hookEventName: "SessionStart",
    additionalContext: ("Branch: " + $b + " · file modificati: " + $s + " · ultimo run CI: " + $c)
  }
}'

In [ ]:
%%bash
cd /tmp/hooks-demo && chmod +x .claude/hooks/*.sh
.claude/hooks/context.sh

Vedi un JSON con `additionalContext`: branch, numero di file modificati e stato dell'ultima CI. Qui la CI risulta «sconosciuto» perché nel notebook non c'è la CLI `gh` collegata a un repository GitHub.

## 5. Hook `Stop`: non chiudere il turno se i test falliscono

In [ ]:
%%writefile /tmp/hooks-demo/.claude/hooks/gate.sh
#!/usr/bin/env bash
# Impedisce la fine del turno se i test non passano.
# set -uo (senza -e): un comando che fallisce NON deve uccidere lo script,
# perché il fallimento è proprio il caso che vogliamo gestire noi.
set -uo pipefail

# Solo se qualcosa è cambiato: un hook Stop che lancia la suite dopo
# ogni domanda è insopportabile. git diff --quiet esce con 0 se non ci
# sono modifiche (--cached = quelle già in staging): se entrambi dicono
# "niente di nuovo", usciamo subito con 0 = via libera.
git diff --quiet && git diff --cached --quiet && exit 0

# Lancia i test catturando tutto l'output nella variabile OUT
if ! OUT=$(npm test --silent 2>&1); then
  # Test falliti: rimanda a Claude le ultime 30 righe dell'output
  # (>&2 = sul canale degli errori, quello che gli hook riportano)
  printf '%s\n' "$OUT" | tail -30 >&2
  exit 2   # exit 2 su un hook Stop = "non chiudere il turno, continua a lavorare"
fi

In [ ]:
%%bash
cd /tmp/hooks-demo && chmod +x .claude/hooks/*.sh
echo "// base" > app.js
git add -A && git commit -qm "base"
echo '--- niente di nuovo da verificare ---'
.claude/hooks/gate.sh 2>&1; echo "exit code: $?"
echo '--- modifica + test che fallisce ---'
echo "// modifica" >> app.js
.claude/hooks/gate.sh 2>&1; echo "exit code: $?"

Prima volta: nessuna modifica rispetto all'ultimo commit, exit 0 senza output. Seconda: il test fallisce, le righe finali dell'output vanno su stderr ed exit **2** significa «non chiudere il turno, continua a lavorare».